# Notebook #29 — Live Trades Replay vs Backtest (Bybit)

Compares the live bot's signals + entries + exits (recorded into
`logs/bybit_bot/<SYMBOL>-YYYY-MM-DD.jsonl`) against a **replay backtest** that
feeds the exact same winner config + the symbol's CSV data into
`_strategy_lib.run_strategy()`. Anything the live bot did that the replay didn't
(or vice versa) flags a parity bug.

## Inputs
- Live log lines       — `logs/bybit_bot/<SYM>-YYYY-MM-DD.jsonl`
- Per-symbol config    — `results/_top_per_symbol/<SYM>/config.json`
- Symbol price history — `notebooks/data/<SYM>/{M5,H1,D1}/ohlcv.csv`

## Outputs (`notebooks/data/`)
- `live_signals_<from>_<to>.csv`   — every signal the bot emitted in the window
- `live_orders_<from>_<to>.csv`    — every `market_order_placed` event
- `live_closes_<from>_<to>.csv`    — every `position_closed` event
- `replay_signals_<from>_<to>.csv` — signals the backtest would have produced
- `replay_vs_live_diff_<from>_<to>.csv` — joined view: matched, live-only, replay-only
- `replay_summary_<from>_<to>.csv` — per-symbol win-rate / net-R / parity stats

Set the window in **Section 1**; everything else is automatic.

## Workflow
1. Edit `WINDOW_FROM` / `WINDOW_TO` and (optionally) `SYMBOLS`.
2. Run all cells.
3. Inspect the `replay_vs_live_diff` table for matched / live-only / replay-only rows.
4. For each mismatch, look at the diagnostics in the live `cycle` log line
   (`logs/bybit_bot/<SYM>-YYYY-MM-DD.jsonl`) and the matching bar in the
   replay backtest's `df_sig` to see *which gate* disagreed.


In [33]:
from __future__ import annotations
import warnings; warnings.filterwarnings('ignore')
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', '{:.6f}'.format)

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    REPO = ROOT.parent
else:
    REPO = ROOT
NB_DIR  = REPO / 'notebooks'
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

from _strategy_lib import (
    run_strategy, stats,
    strategy_trend_pullback, strategy_bb_revert_midline,
    strategy_rsi_extreme_reversal, strategy_donchian_breakout,
    strategy_macd_pullback, strategy_ichimoku, strategy_fib_pullback,
    strategy_sr_zone_bounce, strategy_vwap_reaction, strategy_ema_cross,
)

STRATEGY_FN = {
    'trend_pullback':  strategy_trend_pullback,
    'bb_revert_mid':   strategy_bb_revert_midline,
    'rsi_extreme':     strategy_rsi_extreme_reversal,
    'donchian_brkout': strategy_donchian_breakout,
    'macd_pullback':   strategy_macd_pullback,
    'ichimoku':        strategy_ichimoku,
    'fib_pullback':    strategy_fib_pullback,
    'sr_zone_bounce':  strategy_sr_zone_bounce,
    'vwap_reaction':   strategy_vwap_reaction,
    'ema_cross':       strategy_ema_cross,
}
USE_DAILY = {'trend_pullback': True}

BOT_LOG_DIR = REPO / 'logs' / 'bybit_bot'
WINNERS_DIR = NB_DIR / 'results' / '_top_per_symbol'
DATA_DIR    = NB_DIR / 'data'
OUT_DIR     = NB_DIR / 'data'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ──────────────── EDIT THIS BLOCK ─────────────────
WINDOW_FROM = '2024-05-30'          # inclusive (UTC date)
WINDOW_TO   = '2026-06-17'          # exclusive (UTC date)
SYMBOLS     = ["XLMUSDT", "BCHUSDT","ETHUSDT","BNBUSDT","XRPUSDT","ADAUSDT","SOLUSDT","DOGEUSDT","DOTUSDT","AVAXUSDT","SHIB1000USDT","LTCUSDT"]                  # None = every symbol that has a logs dir entry
BAR_TOLERANCE_MIN = 5               # treat replay vs live signals as 'matched'
                                    # when the bar times are within this many minutes
# ──────────────────────────────────────────────────

T_FROM = pd.Timestamp(WINDOW_FROM)
T_TO   = pd.Timestamp(WINDOW_TO)
TAG    = f"{WINDOW_FROM}_{WINDOW_TO}".replace('-', '')
print(f'Window: {WINDOW_FROM} → {WINDOW_TO}')
print(f'Bot logs dir: {BOT_LOG_DIR}')
print(f'Winners dir : {WINNERS_DIR}')


Window: 2024-05-30 → 2026-06-17
Bot logs dir: D:\bot\ema-h1trend-exchange\logs\bybit_bot
Winners dir : D:\bot\ema-h1trend-exchange\notebooks\results\_top_per_symbol


In [34]:
# ── Exit-model toggle ────────────────────────────────────────────────────────
# The LIVE BOT has no time-based exit — it places SL/TP on the order and only
# closes when one fills (app/run_multi_scalper_bybit.py never calls its `max_hold`).
# So to compare apples-to-apples we disable the replay's time-exit too:
#   None  -> no time-exit, hold to SL/TP   (matches the live bot — default)
#   <int> -> force a fixed bar cap          (old backtest used 96 on M5 / 48 on H1,
#                                            which churned ~2.5x more trades than the bot)
REPLAY_MAX_HOLD_BARS = None
print('Replay time-exit:', 'OFF (match bot, hold to SL/TP)' if REPLAY_MAX_HOLD_BARS is None else f'{REPLAY_MAX_HOLD_BARS} bars')


Replay time-exit: OFF (match bot, hold to SL/TP)


## Section 2 — Discover symbols that have either a live log or a winner config

We auto-pick the set of symbols based on what's on disk. Override via the `SYMBOLS`
constant above if you want a subset.

In [35]:
def list_logged_symbols() -> set[str]:
    if not BOT_LOG_DIR.exists():
        return set()
    syms = set()
    for p in BOT_LOG_DIR.glob('*.jsonl'):
        # filename pattern: SYMBOL-YYYY-MM-DD.jsonl  ; strip the date suffix
        stem = p.stem
        if stem.startswith('_'):
            continue
        parts = stem.rsplit('-', 3)
        sym = parts[0] if len(parts) >= 4 else stem
        syms.add(sym)
    return syms

def list_winner_symbols() -> set[str]:
    if not WINNERS_DIR.exists():
        return set()
    return {p.name for p in WINNERS_DIR.iterdir()
            if p.is_dir() and not p.name.startswith('_') and (p / 'config.json').exists()}

logged  = list_logged_symbols()
winners = list_winner_symbols()
active  = sorted(SYMBOLS) if SYMBOLS else sorted(logged | winners)
print(f'Symbols with live logs    : {len(logged):3d}  → {sorted(logged)}')
print(f'Symbols with winner config: {len(winners):3d}  → {sorted(winners)}')
print(f'Symbols to inspect        : {len(active):3d}  → {active}')


Symbols with live logs    :  12  → ['ADAUSDT', 'AVAXUSDT', 'BCHUSDT', 'BNBUSDT', 'DOGEUSDT', 'DOTUSDT', 'ETHUSDT', 'LTCUSDT', 'SHIB1000USDT', 'SOLUSDT', 'XLMUSDT', 'XRPUSDT']
Symbols with winner config:  14  → ['ADAUSDT', 'AVAXUSDT', 'BCHUSDT', 'BNBUSDT', 'BTCUSDT', 'DOGEUSDT', 'DOTUSDT', 'ETHUSDT', 'LTCUSDT', 'SHIB1000USDT', 'SOLUSDT', 'XAUUSDT', 'XLMUSDT', 'XRPUSDT']
Symbols to inspect        :  12  → ['ADAUSDT', 'AVAXUSDT', 'BCHUSDT', 'BNBUSDT', 'DOGEUSDT', 'DOTUSDT', 'ETHUSDT', 'LTCUSDT', 'SHIB1000USDT', 'SOLUSDT', 'XLMUSDT', 'XRPUSDT']


## Section 3 — Parse live JSONL logs

Each event lives on its own line. We extract three views:

- `signals`    — `event == 'signal'`
- `orders`     — `event == 'market_order_placed'`
- `closes`     — `event == 'position_closed'`

Each row carries the event timestamp + the diagnostics the bot wrote when it
decided. Multi-day logs are concatenated.

In [36]:
def parse_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return rows

def _ts_naive(x):
    """Parse any timestamp string/object to tz-naive UTC pd.Timestamp."""
    try:
        t = pd.Timestamp(x)
    except Exception:
        return pd.NaT
    if t is pd.NaT: return pd.NaT
    if t.tzinfo is not None:
        t = t.tz_convert('UTC').tz_localize(None)
    return t

def load_live(symbol: str, t_from: pd.Timestamp, t_to: pd.Timestamp) -> dict:
    sigs, orders, closes, cycles = [], [], [], []
    if not BOT_LOG_DIR.exists():
        return dict(signals=pd.DataFrame(), orders=pd.DataFrame(),
                    closes=pd.DataFrame(), cycles=pd.DataFrame())
    files = sorted(BOT_LOG_DIR.glob(f'{symbol}-*.jsonl'))
    for fp in files:
        for ev in parse_jsonl(fp):
            ts = _ts_naive(ev.get('ts', ''))
            if ts is pd.NaT or ts < t_from or ts >= t_to: continue
            kind = ev.get('event')
            ev['ts'] = ts
            if   kind == 'signal':                sigs.append(ev)
            elif kind == 'market_order_placed':   orders.append(ev)
            elif kind == 'position_closed':       closes.append(ev)
            elif kind == 'cycle':                 cycles.append(ev)
    return dict(
        signals=pd.DataFrame(sigs),
        orders=pd.DataFrame(orders),
        closes=pd.DataFrame(closes),
        cycles=pd.DataFrame(cycles),
    )

live_by_sym: dict[str, dict] = {}
for sym in active:
    live_by_sym[sym] = load_live(sym, T_FROM, T_TO)
n_total_sigs   = sum(len(v['signals']) for v in live_by_sym.values())
n_total_orders = sum(len(v['orders'])  for v in live_by_sym.values())
n_total_closes = sum(len(v['closes'])  for v in live_by_sym.values())
print(f'Loaded live events — signals={n_total_sigs}  orders={n_total_orders}  closes={n_total_closes}')
for sym in active:
    v = live_by_sym[sym]
    print(f'  {sym:14}  signals={len(v["signals"]):4d}  orders={len(v["orders"]):4d}  closes={len(v["closes"]):4d}  cycles={len(v["cycles"]):5d}')


KeyboardInterrupt: 

## Section 4 — Replay backtest on the same window

We invoke `run_strategy()` from `_strategy_lib.py` with each symbol's winner config and
filter the resulting trades to the live window. **This is the same backtest engine the
sweep used** — identical fees, identical SL/TP, identical signal pipeline.

In [ ]:
def replay_one(symbol: str, window_from: pd.Timestamp, window_to: pd.Timestamp) -> dict:
    cfg_path = WINNERS_DIR / symbol / 'config.json'
    if not cfg_path.exists():
        return dict(skipped=f'no winner config for {symbol}')
    cfg = json.loads(cfg_path.read_text(encoding='utf-8'))
    sname = cfg['strategy']
    fn = STRATEGY_FN.get(sname)
    if fn is None:
        return dict(skipped=f'unknown strategy {sname}')
    params = dict(cfg.get('params', {}))
    if 'session' in params and isinstance(params['session'], list):
        params['session'] = tuple(params['session'])
    if 'confirms' in params and isinstance(params['confirms'], list):
        params['confirms'] = tuple(params['confirms'])
    tf  = cfg['base_tf']
    htf = cfg.get('htf', 'H1' if tf == 'M5' else 'H4')
    use_daily = bool(cfg.get('use_daily', USE_DAILY.get(sname, False)))
    rr = float(cfg.get('rr', 0.0) or 0.0)
    # We pad backwards 60 days for indicator warmup so the early signals
    # use the same RSI/ATR state the live bot saw.
    pad_from = (window_from - pd.Timedelta(days=60)).strftime('%Y-%m-%d')
    date_to  = window_to.strftime('%Y-%m-%d')
    # Match the live bot's exit model (no time-exit — see the toggle cell after
    # Section 1). None -> hold to SL/TP like the bot; an int re-imposes a bar cap.
    _mh = 10**9 if REPLAY_MAX_HOLD_BARS is None else int(REPLAY_MAX_HOLD_BARS)
    kw = dict(use_daily=use_daily, max_hold_bars=_mh,
              date_from=pad_from, date_to=date_to)
    if rr > 0:
        kw['rr'] = rr
    trades, df_sig = run_strategy(symbol, tf, htf, fn, params, **kw)
    if df_sig.empty:
        return dict(skipped=f'no data for {symbol}')
    # Trade rows that opened inside the live window
    rows = []
    for t in trades:
        if t.entry_time is None: continue
        et = pd.Timestamp(t.entry_time)
        if et < window_from or et >= window_to: continue
        rows.append({
            'entry_time': et, 'side': 'long' if t.side == 1 else 'short',
            'entry': t.entry, 'sl': t.sl, 'tp': t.tp,
            'exit_time': pd.Timestamp(t.exit_time) if t.exit_time is not None else pd.NaT,
            'exit': t.exit, 'reason': t.reason,
            'R': round(t.r_multiple, 3), 'net_R': round(t.net_r, 3),
        })
    trades_df = pd.DataFrame(rows)
    # Every bar where df_sig.signal != 0 inside the window (signals, not just trades)
    df_sig = df_sig.copy()
    df_sig['time'] = pd.to_datetime(df_sig['time'])
    sig_rows = df_sig[(df_sig.get('signal', 0) != 0) &
                      (df_sig['time'] >= window_from) &
                      (df_sig['time'] <  window_to)]
    signals_df = sig_rows[['time', 'signal', 'close', 'sl_price']].copy()
    if 'tp_price' in sig_rows.columns:
        signals_df['tp_price'] = sig_rows['tp_price']
    return dict(trades=trades_df, signals=signals_df, cfg=cfg)

replay_by_sym: dict[str, dict] = {}
for sym in active:
    try:
        replay_by_sym[sym] = replay_one(sym, T_FROM, T_TO)
    except Exception as exc:
        replay_by_sym[sym] = {'skipped': f'exception: {exc!r}'}
for sym, r in replay_by_sym.items():
    if 'skipped' in r:
        print(f'  {sym:14}  SKIP — {r["skipped"]}')
    else:
        print(f'  {sym:14}  replay  signals={len(r["signals"]):4d}  trades={len(r["trades"]):4d}')


  ADAUSDT         replay  signals=7295  trades= 146
  AVAXUSDT        replay  signals=7858  trades= 156
  BCHUSDT         replay  signals= 825  trades=  36
  BNBUSDT         replay  signals= 366  trades= 103
  DOGEUSDT        replay  signals=7138  trades= 168
  DOTUSDT         replay  signals=8050  trades= 140
  ETHUSDT         replay  signals=1101  trades=  51
  LTCUSDT         replay  signals=1157  trades=  15
  SHIB1000USDT    replay  signals=8032  trades= 118
  SOLUSDT         replay  signals=9714  trades= 377
  XLMUSDT         replay  signals=1367  trades=  32
  XRPUSDT         replay  signals=1218  trades=  39


## Section 5 — Match live ⇄ replay signals

For each symbol we join the live `signal` events with replay-produced signals by
(symbol, bar_time, direction). Categorise each row as:

- `matched`      — live + replay agree on the bar and direction.
- `live_only`    — bot fired a signal the backtest didn't reproduce. → drift bug.
- `replay_only`  — backtest expected a signal the bot missed.        → live miss / outage.

All matched / mismatched rows get a `delta_entry` (live entry − replay close)
and a `delta_sl/tp` column so price-level drift is obvious.

In [ ]:
def _to_ts(x):
    try:
        t = pd.Timestamp(x)
    except Exception:
        return pd.NaT
    if t is pd.NaT: return pd.NaT
    if t.tzinfo is not None:
        t = t.tz_convert('UTC').tz_localize(None)
    return t

def signals_diff(symbol: str) -> pd.DataFrame:
    live = live_by_sym.get(symbol, {}).get('signals', pd.DataFrame())
    rep  = replay_by_sym.get(symbol, {}).get('signals', pd.DataFrame())
    if live.empty and rep.empty:
        return pd.DataFrame()
    if not live.empty:
        live = live.copy()
        live['bar_dt']    = live['bar_time'].apply(_to_ts)
        live['dir_str']   = live['direction']
        live['side_int']  = np.where(live['direction'] == 'long', 1, -1)
    if not rep.empty:
        rep = rep.copy()
        rep['bar_dt']    = rep['time'].apply(_to_ts)
        rep['side_int']  = rep['signal'].astype(int)
        rep['dir_str']   = np.where(rep['side_int'] == 1, 'long', 'short')

    tol = pd.Timedelta(minutes=BAR_TOLERANCE_MIN)
    rows = []
    used_rep = set()
    if not live.empty:
        for _, lr in live.iterrows():
            match = None
            if not rep.empty:
                cand = rep[(rep['side_int'] == lr['side_int']) &
                           (rep['bar_dt'] >= lr['bar_dt'] - tol) &
                           (rep['bar_dt'] <= lr['bar_dt'] + tol)]
                cand = cand[~cand.index.isin(used_rep)]
                if not cand.empty:
                    match = cand.iloc[0]
                    used_rep.add(match.name)
            rows.append({
                'symbol':       symbol,
                'kind':         'matched' if match is not None else 'live_only',
                'bar_time':     lr['bar_dt'],
                'direction':    lr['dir_str'],
                'live_entry':   float(lr.get('entry', np.nan)),
                'live_sl':      float(lr.get('sl',    np.nan)),
                'live_tp':      float(lr.get('tp',    np.nan)),
                'rep_close':    float(match['close'])    if match is not None else np.nan,
                'rep_sl':       float(match['sl_price']) if match is not None else np.nan,
                'rep_tp':       float(match['tp_price']) if (match is not None and 'tp_price' in match.index) else np.nan,
            })
    if not rep.empty:
        unmatched = rep[~rep.index.isin(used_rep)]
        for _, rr in unmatched.iterrows():
            rows.append({
                'symbol':       symbol,
                'kind':         'replay_only',
                'bar_time':     rr['bar_dt'],
                'direction':    rr['dir_str'],
                'live_entry':   np.nan,
                'live_sl':      np.nan,
                'live_tp':      np.nan,
                'rep_close':    float(rr['close']),
                'rep_sl':       float(rr['sl_price']),
                'rep_tp':       float(rr['tp_price']) if 'tp_price' in rr.index else np.nan,
            })
    out = pd.DataFrame(rows).sort_values('bar_time') if rows else pd.DataFrame()
    if not out.empty:
        out['delta_entry'] = out['live_entry'] - out['rep_close']
        out['delta_sl']    = out['live_sl']    - out['rep_sl']
        out['delta_tp']    = out['live_tp']    - out['rep_tp']
    return out

diff_frames = [signals_diff(sym) for sym in active]
diff_all = pd.concat([d for d in diff_frames if not d.empty], ignore_index=True) if diff_frames else pd.DataFrame()
print(f'Total diff rows: {len(diff_all)}')
if not diff_all.empty:
    print(diff_all['kind'].value_counts())
    diff_all.head(40)


Total diff rows: 55072
kind
replay_only    53205
live_only        951
matched          916
Name: count, dtype: int64


## Section 6 — Per-symbol parity scoreboard

How well did live track replay over the window? For each symbol we report
`matched / live_only / replay_only` signal counts + the win rate / net of the
**live actuals** versus the **replay backtest**.

> ⚠️ **PnL source fix.** The live win-rate/PnL here is **no longer** read from the
> bot's local `position_closed` logs. Those are corrupted: when the bot's
> secondary REST poll for the exit price came back empty it wrote
> `exit_price=NaN, pnl_usdt=0`, silently logging real **wins** as break-even
> (~1/3 of closes). So the local log understates wins *and* net PnL.
>
> `live_*` now comes from **Bybit's authoritative closed-PnL ledger**, cached by
> notebook **#01** to `data/<SYM>/trades/closed_pnl_history.csv` (or rebuild via
> `scripts/reconstruct_pnl.py`). Run notebook #01 first to refresh that cache.
> The `log_*` columns keep the corrupted local-log numbers *only* so the
> discrepancy is visible. (Section 13's `live_pnl_usdt` still reads the local
> log and carries the same caveat.)

In [ ]:
# ── Authoritative live PnL ──────────────────────────────────────────────────
# True realised PnL comes from Bybit's closed-PnL ledger, NOT the local
# position_closed logs (whose pnl_usdt is zeroed whenever the exit-price reconcile
# poll came back empty — see Section 6 header). We read the cache notebook #01
# writes to data/<SYM>/trades/closed_pnl_history.csv.
def load_authoritative_closes(symbol: str, t_from: pd.Timestamp, t_to: pd.Timestamp) -> pd.DataFrame:
    fp = DATA_DIR / symbol / 'trades' / 'closed_pnl_history.csv'
    if not fp.exists():
        return pd.DataFrame()
    try:
        df = pd.read_csv(fp)
    except pd.errors.EmptyDataError:   # symbol traded 0 closes → empty cache file
        return pd.DataFrame()
    if df.empty or 'closedPnl' not in df.columns or 'exit_time' not in df.columns:
        return pd.DataFrame()
    df['exit_time'] = pd.to_datetime(df['exit_time'], utc=True, errors='coerce').dt.tz_localize(None)
    df['closedPnl'] = pd.to_numeric(df['closedPnl'], errors='coerce')
    df = df.dropna(subset=['exit_time', 'closedPnl'])
    return df[(df['exit_time'] >= t_from) & (df['exit_time'] < t_to)]

_missing_cache = []
rows = []
for sym in active:
    diff = next((d for d in diff_frames if not d.empty and d['symbol'].iloc[0] == sym), pd.DataFrame())
    n_matched     = int((diff['kind'] == 'matched').sum())     if not diff.empty else 0
    n_live_only   = int((diff['kind'] == 'live_only').sum())   if not diff.empty else 0
    n_replay_only = int((diff['kind'] == 'replay_only').sum()) if not diff.empty else 0

    # ── live actuals: authoritative Bybit closed-PnL ledger ──
    auth = load_authoritative_closes(sym, T_FROM, T_TO)
    live_n    = len(auth)
    live_wins = int((auth['closedPnl'] > 0).sum()) if live_n else 0
    live_net  = float(auth['closedPnl'].sum())     if live_n else 0.0

    # ── local-log numbers kept ONLY to expose the discrepancy ──
    local_closes = live_by_sym.get(sym, {}).get('closes', pd.DataFrame())
    if not local_closes.empty and 'pnl_usdt' in local_closes.columns:
        log_n   = len(local_closes)
        log_net = float(local_closes['pnl_usdt'].astype(float).sum())
        if live_n == 0:
            _missing_cache.append(sym)
    else:
        log_n, log_net = 0, 0.0

    rep_trades = replay_by_sym.get(sym, {}).get('trades', pd.DataFrame())
    if not rep_trades.empty:
        rep_wins  = int((rep_trades['R'] > 0).sum())
        rep_n     = len(rep_trades)
        rep_net_R = float(rep_trades['net_R'].sum())
    else:
        rep_wins, rep_n, rep_net_R = 0, 0, 0.0

    rows.append({
        'symbol': sym,
        'live_closed':   live_n, 'live_wins':   live_wins,
        'live_WR_%':     round(100 * live_wins / live_n, 1) if live_n else 0.0,
        'live_net_USDT': round(live_net, 2),
        'log_closed':    log_n,  'log_net_USDT': round(log_net, 2),   # corrupted local log — reference only
        'replay_trades': rep_n, 'replay_wins': rep_wins,
        'replay_WR_%':   round(100 * rep_wins / rep_n, 1) if rep_n else 0.0,
        'replay_net_R':  round(rep_net_R, 3),
        'matched':      n_matched,
        'live_only':    n_live_only,
        'replay_only':  n_replay_only,
    })
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

# Totals + an explicit local-log-vs-truth discrepancy line.
tot_live_n   = int(summary['live_closed'].sum())
tot_live_w   = int(summary['live_wins'].sum())
tot_live_net = float(summary['live_net_USDT'].sum())
tot_log_net  = float(summary['log_net_USDT'].sum())
print(f"\nLIVE ACTUALS (Bybit ledger): {tot_live_n} closes  "
      f"{tot_live_w}W  WR {100*tot_live_w/tot_live_n:.1f}%  net {tot_live_net:+.2f} USDT"
      if tot_live_n else "\nLIVE ACTUALS: no cached closed-PnL found.")
print(f"Local position_closed log  : net {tot_log_net:+.2f} USDT  "
      f"(understated by {tot_live_net - tot_log_net:+.2f} USDT — the NaN/pnl=0 bug)")
if _missing_cache:
    print(f"\n⚠ No cached closed-PnL for {_missing_cache} — run notebook #01 to refresh "
          f"data/<SYM>/trades/closed_pnl_history.csv before trusting their live_* = 0.")

      symbol  live_closed  live_wins  live_WR_%  live_net_USDT  log_closed  log_net_USDT  replay_trades  replay_wins  replay_WR_%  replay_net_R  matched  live_only  replay_only
     ADAUSDT           20          8  40.000000     115.750000          15   -137.640000            146           45    30.800000      0.965000      160          1         7135
    AVAXUSDT           25          9  36.000000      68.750000          24   -128.720000            156           54    34.600000     21.219000      124          2         7734
     BCHUSDT            3          3 100.000000     131.590000           2      0.000000             36           10    27.800000     -1.985000       17        155          808
     BNBUSDT            0          0   0.000000       0.000000           0      0.000000            103           49    47.600000     37.965000        0         13          366
    DOGEUSDT           17          4  23.500000    -101.890000          14   -284.530000            168           6

## Section 7 — Persist artefacts

Everything saves to `notebooks/data/` tagged with the window so multiple replays don't
stomp each other.

In [ ]:
def concat_live(field: str) -> pd.DataFrame:
    frames = []
    for sym, v in live_by_sym.items():
        df = v.get(field, pd.DataFrame())
        if df.empty: continue
        df = df.copy(); df['symbol'] = sym
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

live_signals = concat_live('signals')
live_orders  = concat_live('orders')
live_closes  = concat_live('closes')

replay_signals_all = pd.concat(
    [r['signals'].assign(symbol=s) for s, r in replay_by_sym.items()
     if not r.get('skipped') and not r['signals'].empty],
    ignore_index=True
) if any(not r.get('skipped') for r in replay_by_sym.values()) else pd.DataFrame()

out_files = {
    f'live_signals_{TAG}.csv':         live_signals,
    f'live_orders_{TAG}.csv':          live_orders,
    f'live_closes_{TAG}.csv':          live_closes,
    f'replay_signals_{TAG}.csv':       replay_signals_all,
    f'replay_vs_live_diff_{TAG}.csv':  diff_all,
    f'replay_summary_{TAG}.csv':       summary,
}
for fname, df in out_files.items():
    if df.empty:
        print(f'  {fname:42s}  (empty — not written)')
        continue
    p = OUT_DIR / fname
    df.to_csv(p, index=False)
    print(f'  {fname:42s}  rows={len(df):5d}  -> {p}')


  live_signals_20240530_20260617.csv          rows= 1867  -> D:\bot\ema-h1trend-exchange\notebooks\data\live_signals_20240530_20260617.csv
  live_orders_20240530_20260617.csv           rows=  112  -> D:\bot\ema-h1trend-exchange\notebooks\data\live_orders_20240530_20260617.csv
  live_closes_20240530_20260617.csv           rows=  101  -> D:\bot\ema-h1trend-exchange\notebooks\data\live_closes_20240530_20260617.csv
  replay_signals_20240530_20260617.csv        rows=54121  -> D:\bot\ema-h1trend-exchange\notebooks\data\replay_signals_20240530_20260617.csv
  replay_vs_live_diff_20240530_20260617.csv   rows=55072  -> D:\bot\ema-h1trend-exchange\notebooks\data\replay_vs_live_diff_20240530_20260617.csv
  replay_summary_20240530_20260617.csv        rows=   12  -> D:\bot\ema-h1trend-exchange\notebooks\data\replay_summary_20240530_20260617.csv


## Section 8 — Drilldown on a single mismatch

Pick a row from `replay_vs_live_diff_*.csv` and look at the *full cycle log* the bot
wrote around that bar. The `cycle` event contains every gate value the strategy fn
evaluated (RSI/ADX/h1_trend/MACD), so any disagreement vs the replay's `df_sig` row
tells you *which gate* drifted (data freshness, indicator value, etc.).

In [ ]:
def explain(sym: str, bar_time_str: str, window_min: int = 30) -> pd.DataFrame:
    cycles = live_by_sym.get(sym, {}).get('cycles', pd.DataFrame())
    if cycles.empty:
        return pd.DataFrame()
    bar = pd.Timestamp(bar_time_str)
    cycles = cycles.copy()
    cycles['ts'] = pd.to_datetime(cycles['ts'])
    near = cycles[(cycles['ts'] >= bar - pd.Timedelta(minutes=window_min)) &
                   (cycles['ts'] <= bar + pd.Timedelta(minutes=window_min))]
    return near

# Example: change SYM and TIME below to inspect the gates the bot saw.
if not diff_all.empty and (diff_all['kind'] != 'matched').any():
    first = diff_all[diff_all['kind'] != 'matched'].iloc[0]
    print(f'Drilldown candidate: {first["symbol"]} bar={first["bar_time"]}  kind={first["kind"]}')
    near = explain(first['symbol'], str(first['bar_time']))
    if near.empty:
        print('No cycle events recorded near this time. Was the bot running?')
    else:
        # Keep the columns most useful for forensic comparison.
        keep = [c for c in ('ts','last_bar','m5_bars','htf_bars','age_min',
                              'signal','diag') if c in near.columns]
        print(near[keep].to_string(index=False))
else:
    print('Either no diffs found, or all rows are "matched". Pick a different window.')


Drilldown candidate: ADAUSDT bar=2025-06-24 14:40:00  kind=replay_only
No cycle events recorded near this time. Was the bot running?


## Section 9 — Detailed replay ledger

Reads the `replay_by_sym` dict that Section 4 built and expands each replay trade into a full row with `quantity`, `risk_usdt`, `pnl_usdt`, `pnl_r`, and a normalised `exit_reason`. The trades come from `_strategy_lib.run_strategy()` — the same engine the live bot uses for signal generation and SL/TP.

`risk_usdt` defaults to 20.0 (the live bot's `RISK_FIXED_USDT`, i.e. 1R = 20 USDT).

> **`pnl_usdt` is fee-adjusted.** The engine already nets Bybit's ~0.11% round-trip
> taker fee into `net_R`, so the realistic USDT figure is `net_R × risk_usdt`.
> (The old cell multiplied the *raw* price move and silently dropped the fee, which
> inflated the replay total — e.g. +846 gross vs **+548 net**.) The raw figure is
> kept as `pnl_usdt_gross` for reference. Set `SLIPPAGE_BPS_ROUND_TRIP` below to
> model execution slippage on top of fees and watch the total move toward the live
> ledger's +446.

In [ ]:
# ──────────── configurable ────────────
RISK_USDT = 20.0                    # matches live bot RISK_FIXED_USDT (1R in USDT)
SLIPPAGE_BPS_ROUND_TRIP = 0.0       # extra execution slippage to model, in bps of
                                    # notional per round-trip. 0 = fees only (the
                                    # engine already nets the 0.11% taker fee into
                                    # net_R). Try ~2-4 to approach real fills.
# ──────────────────────────────────────

EXIT_REASON_MAP = {'tp': 'TP', 'sl': 'SL', 'time': 'TIME'}

def _expand_replay_trades(symbol: str, repl: dict) -> list[dict]:
    trades = repl.get('trades', pd.DataFrame())
    if trades.empty:
        return []
    rows = []
    for _, t in trades.iterrows():
        entry = float(t['entry']); sl = float(t['sl']); tp = float(t['tp'])
        exit_ = float(t['exit'])
        risk_per_unit = abs(entry - sl)
        qty = (RISK_USDT / risk_per_unit) if risk_per_unit > 0 else 0.0
        side_int = 1 if t['side'] == 'long' else -1

        # Gross (idealised) USDT straight from the raw price move — kept for reference.
        pnl_usdt_gross = round(qty * (exit_ - entry) * side_int, 4)

        # Realistic USDT: the engine's fee-adjusted net_R scaled to the risk budget
        # (1R = RISK_USDT), minus optional execution slippage on the notional.
        net_r = float(t.get('net_R', t.get('R', 0.0)))
        slippage_usdt = (SLIPPAGE_BPS_ROUND_TRIP / 1e4) * qty * entry
        pnl_usdt = round(net_r * RISK_USDT - slippage_usdt, 4)

        reason_raw = str(t.get('reason', ''))
        rows.append({
            'symbol':         symbol,
            'entry_time_utc': pd.Timestamp(t['entry_time']),
            'exit_time_utc':  pd.Timestamp(t['exit_time']) if pd.notna(t['exit_time']) else pd.NaT,
            'side':           t['side'],
            'entry_price':    entry,
            'exit_price':     exit_,
            'stop_loss':      sl,
            'take_profit':    tp,
            'quantity':       round(qty, 6),
            'risk_usdt':      RISK_USDT,
            'pnl_usdt':       pnl_usdt,            # fee-adjusted (realistic)
            'pnl_usdt_gross': pnl_usdt_gross,      # raw price move (idealised, no fees)
            'pnl_r':          round(net_r, 4),
            'exit_reason':    EXIT_REASON_MAP.get(reason_raw, reason_raw),
            'signal_bar_utc': pd.Timestamp(t['entry_time']),
        })
    return rows

_replay_rows = []
for sym in active:
    repl = replay_by_sym.get(sym, {})
    if 'skipped' in repl:
        continue
    _replay_rows.extend(_expand_replay_trades(sym, repl))

replay_trades_df = pd.DataFrame(_replay_rows)
if not replay_trades_df.empty:
    replay_trades_df = replay_trades_df.sort_values(
        ['symbol', 'entry_time_utc']).reset_index(drop=True)

print(f'Replay trades total: {len(replay_trades_df)}')
if not replay_trades_df.empty:
    g = replay_trades_df['pnl_usdt_gross'].sum()
    n = replay_trades_df['pnl_usdt'].sum()
    print(f'Replay net USDT  : {n:+.2f}  (fee-adjusted, SLIPPAGE_BPS_ROUND_TRIP={SLIPPAGE_BPS_ROUND_TRIP})')
    print(f'Replay gross USDT: {g:+.2f}  (raw price move, no fees — the old inflated figure)')
    print(f'Fees+slippage drag: {n - g:+.2f} USDT')
replay_trades_df.tail(20)

Replay trades total: 1381
Replay net USDT  : +2905.50  (fee-adjusted, SLIPPAGE_BPS_ROUND_TRIP=0.0)
Replay gross USDT: +4880.00  (raw price move, no fees — the old inflated figure)
Fees+slippage drag: -1974.50 USDT


,symbol,entry_time_utc,exit_time_utc,side,entry_price,exit_price,stop_loss,take_profit,quantity,risk_usdt,pnl_usdt,pnl_usdt_gross,pnl_r,exit_reason,signal_bar_utc
1361,XRPUSDT,2025-06-23 21:00:00,2025-06-30 19:00:00,long,2.075400,2.253616,1.956589,2.253616,168.334661,20.000000,29.620000,30.000000,1.481000,TP,2025-06-23 21:00:00
1362,XRPUSDT,2025-06-30 20:00:00,2025-07-01 19:00:00,long,2.319200,2.157664,2.157664,2.561504,123.811515,20.000000,-20.320000,-20.000000,-1.016000,SL,2025-06-30 20:00:00
1363,XRPUSDT,2025-07-03 10:00:00,2025-07-04 07:00:00,long,2.291400,2.221386,2.221386,2.396421,285.657911,20.000000,-20.720000,-20.000000,-1.036000,SL,2025-07-03 10:00:00
1364,XRPUSDT,2025-07-06 07:00:00,2025-07-06 10:00:00,long,2.231600,2.280355,2.199097,2.280355,615.323342,20.000000,28.480000,30.000000,1.424000,TP,2025-07-06 07:00:00
1365,XRPUSDT,2025-07-06 11:00:00,2025-07-09 11:00:00,long,2.274000,2.376590,2.205606,2.376590,292.424861,20.000000,29.260000,30.000000,1.463000,TP,2025-07-06 11:00:00
1366,XRPUSDT,2025-07-09 12:00:00,2025-07-10 18:00:00,long,2.380900,2.510313,2.294624,2.510313,231.815290,20.000000,29.400000,30.000000,1.470000,TP,2025-07-09 12:00:00
1367,XRPUSDT,2025-07-10 19:00:00,2025-07-11 09:00:00,long,2.487900,2.609738,2.406675,2.609738,246.229078,20.000000,29.320000,30.000000,1.466000,TP,2025-07-10 19:00:00
1368,XRPUSDT,2025-07-11 10:00:00,2025-07-11 13:00:00,long,2.613600,2.793911,2.493393,2.793911,166.379427,20.000000,29.520000,30.000000,1.476000,TP,2025-07-11 10:00:00
1369,XRPUSDT,2025-07-11 14:00:00,2025-07-17 08:00:00,long,2.799700,3.218907,2.520228,3.218907,71.563631,20.000000,29.780000,30.000000,1.489000,TP,2025-07-11 14:00:00
1370,XRPUSDT,2025-07-17 09:00:00,2025-07-18 00:00:00,long,3.237500,3.619258,2.982995,3.619258,78.583782,20.000000,29.720000,30.000000,1.486000,TP,2025-07-17 09:00:00


## Section 10 — Persist replay artefacts

Writes the full ledger to `artifacts/replay_trades.csv` and `artifacts/replay_trades.parquet`, plus one CSV per symbol under `artifacts/replay_trades/<SYMBOL>.csv`. The parquet write is best-effort (needs `pyarrow` or `fastparquet`); if it fails the CSV still goes out.

In [ ]:
ARTIFACTS_DIR  = REPO / 'artifacts'
PER_SYMBOL_DIR = ARTIFACTS_DIR / 'replay_trades'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
PER_SYMBOL_DIR.mkdir(parents=True, exist_ok=True)

csv_path     = ARTIFACTS_DIR / 'replay_trades.csv'
parquet_path = ARTIFACTS_DIR / 'replay_trades.parquet'

if not replay_trades_df.empty:
    replay_trades_df.to_csv(csv_path, index=False)
    print(f'  {csv_path}   ({len(replay_trades_df)} rows)')
    try:
        replay_trades_df.to_parquet(parquet_path, index=False)
        print(f'  {parquet_path}   ({len(replay_trades_df)} rows)')
    except Exception as exc:
        print(f'  parquet skipped: {exc}')
    n_files = 0
    for sym, grp in replay_trades_df.groupby('symbol'):
        (PER_SYMBOL_DIR / f'{sym}.csv').write_text(
            grp.to_csv(index=False), encoding='utf-8')
        n_files += 1
    print(f'  {PER_SYMBOL_DIR}/   ({n_files} per-symbol files)')
else:
    print('No replay trades to save in this window.')


  D:\bot\ema-h1trend-exchange\artifacts\replay_trades.csv   (1381 rows)
  D:\bot\ema-h1trend-exchange\artifacts\replay_trades.parquet   (1381 rows)
  D:\bot\ema-h1trend-exchange\artifacts\replay_trades/   (12 per-symbol files)


## Section 11 — Per-symbol replay summary

Trades, wins, losses, win rate, total USDT PnL, average R, and gross profit factor for each symbol that produced at least one replay trade.

In [ ]:
def per_symbol_summary(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    rows = []
    for sym, g in df.groupby('symbol'):
        n = len(g)
        wins = int((g['pnl_usdt'] > 0).sum())
        losses = n - wins
        wr = 100.0 * wins / n if n else 0.0
        win_pnl  = g.loc[g['pnl_usdt'] > 0,  'pnl_usdt'].sum()
        loss_pnl = -g.loc[g['pnl_usdt'] <= 0, 'pnl_usdt'].sum()
        if loss_pnl > 0:
            pf = win_pnl / loss_pnl
        else:
            pf = float('inf') if win_pnl > 0 else 0.0
        rows.append({
            'symbol': sym, 'trades': n, 'wins': wins, 'losses': losses,
            'win_rate_%':     round(wr, 2),
            'total_pnl_usdt': round(float(g['pnl_usdt'].sum()), 2),
            'avg_pnl_r':      round(float(g['pnl_r'].mean()), 3),
            'profit_factor':  round(pf, 3) if pf != float('inf') else pf,
        })
    return pd.DataFrame(rows).sort_values('symbol').reset_index(drop=True)

replay_summary_by_symbol = per_symbol_summary(replay_trades_df)
if replay_summary_by_symbol.empty:
    print('No replay trades — per-symbol summary skipped.')
else:
    print(replay_summary_by_symbol.to_string(index=False))


      symbol  trades  wins  losses  win_rate_%  total_pnl_usdt  avg_pnl_r  profit_factor
     ADAUSDT     146    45     101   30.820000       19.300000   0.007000       1.009000
    AVAXUSDT     156    54     102   34.620000      424.380000   0.136000       1.193000
     BCHUSDT      36    10      26   27.780000      -39.700000  -0.055000       0.926000
     BNBUSDT     103    49      54   47.570000      759.300000   0.369000       1.665000
    DOGEUSDT     168    62     106   36.900000      729.800000   0.217000       1.321000
     DOTUSDT     140    47      93   33.570000      266.280000   0.095000       1.133000
     ETHUSDT      51    20      31   39.220000      351.180000   0.344000       1.550000
     LTCUSDT      15     5      10   33.330000       43.580000   0.145000       1.214000
SHIB1000USDT     118    45      73   38.140000      597.100000   0.253000       1.379000
     SOLUSDT     377   111     266   29.440000     -421.720000  -0.056000       0.927000
     XLMUSDT      32 

## Section 12 — Global replay summary

Totals across every replay trade in the window — total trades, total USDT PnL, total R, win rate, expectancy in R, and the longest losing streak (consecutive trades with `pnl_usdt <= 0`, in time order).

In [ ]:
def max_consecutive_losses(df: pd.DataFrame) -> int:
    if df.empty:
        return 0
    losses_seq = (df.sort_values('entry_time_utc')['pnl_usdt'] <= 0).astype(int).to_numpy()
    best = cur = 0
    for x in losses_seq:
        if x == 1:
            cur += 1
            if cur > best: best = cur
        else:
            cur = 0
    return best

if replay_trades_df.empty:
    print('No replay trades — global summary skipped.')
else:
    n = len(replay_trades_df)
    wins = int((replay_trades_df['pnl_usdt'] > 0).sum())
    wr = 100.0 * wins / n
    total_pnl_usdt = float(replay_trades_df['pnl_usdt'].sum())
    total_pnl_r   = float(replay_trades_df['pnl_r'].sum())
    expectancy_r  = float(replay_trades_df['pnl_r'].mean())
    max_loss_streak = max_consecutive_losses(replay_trades_df)
    global_summary = pd.DataFrame([{
        'total_trades':       n,
        'total_pnl_usdt':     round(total_pnl_usdt, 2),
        'total_pnl_r':        round(total_pnl_r, 3),
        'win_rate_%':         round(wr, 2),
        'expectancy_r':       round(expectancy_r, 4),
        'max_consec_losses':  max_loss_streak,
    }])
    print(global_summary.to_string(index=False))


 total_trades  total_pnl_usdt  total_pnl_r  win_rate_%  expectancy_r  max_consec_losses
         1381     2905.500000   145.275000   34.540000      0.105200                 19


## Section 13 — Live vs replay trade comparison

Matches **authoritative live trades** (Bybit closed-PnL ledger, the same source
Section 6 uses — *not* the corrupted local `position_closed` logs) against the
replay backtest trades, on (symbol, side, entry-time within ±15 min).

Three categories:

- `matched`      — a real trade that the backtest also produced (a backtest twin).
- `live_only`    — a real trade with **no** backtest twin. Usually sequencing
  drift (backtest was still in a position when live was flat) or a real-time
  gate/indicator difference.
- `replay_only`  — a backtest trade never taken live. Usually the bot was
  offline/starting up, was in another position, or missed the signal.

> Note: `live_*` PnL here is the exchange's realised `closedPnl` (truth). The
> replay PnL uses a fixed 20-USDT risk model with idealised bar fills, so even a
> `matched` pair won't have identical PnL — compare **win rate**, not absolute USDT.

In [ ]:
# ── Authoritative live trades (Bybit closed-PnL ledger) ─────────────────────
# Reuses load_authoritative_closes() from Section 6. We additionally parse the
# entry_time + position direction so each real trade can be matched to a replay
# trade. This REPLACES the old path that read the corrupted local position_closed
# logs (whose pnl_usdt is zeroed by the NaN/pnl=0 bug — see Section 6 header).
def _auth_live_trades(symbol: str) -> pd.DataFrame:
    df = load_authoritative_closes(symbol, T_FROM, T_TO)   # filtered by exit_time, closedPnl parsed
    if df.empty:
        return pd.DataFrame()
    return pd.DataFrame({
        'symbol':         symbol,
        'entry_time_utc': pd.to_datetime(df['entry_time'], utc=True, errors='coerce').dt.tz_localize(None),
        'exit_time_utc':  df['exit_time'],   # already tz-naive from load_authoritative_closes
        'side':           df['position'].astype(str).str.lower(),     # 'long' / 'short'
        'entry_price':    pd.to_numeric(df.get('avgEntryPrice'), errors='coerce'),
        'exit_price':     pd.to_numeric(df.get('avgExitPrice'),  errors='coerce'),
        'quantity':       pd.to_numeric(df.get('closedSize'),    errors='coerce'),
        'pnl_usdt':       df['closedPnl'].round(4),
    })

_live_blocks = [_auth_live_trades(sym) for sym in active]
_live_blocks = [b for b in _live_blocks if not b.empty]
live_trades_df = (pd.concat(_live_blocks, ignore_index=True)
                  .dropna(subset=['entry_time_utc'])
                  if _live_blocks else pd.DataFrame())

MATCH_TOL = pd.Timedelta(minutes=15)

def match_live_vs_replay(live: pd.DataFrame, replay: pd.DataFrame) -> pd.DataFrame:
    """Iterate over each REAL trade and look for a backtest twin (greedy, 1:1)."""
    rows, used_rep = [], set()
    if not live.empty:
        for _, l in live.iterrows():
            cand = pd.DataFrame()
            if not replay.empty:
                cand = replay[(replay['symbol'] == l['symbol']) &
                              (replay['side']   == l['side']) &
                              (replay['entry_time_utc'] >= l['entry_time_utc'] - MATCH_TOL) &
                              (replay['entry_time_utc'] <= l['entry_time_utc'] + MATCH_TOL)]
                cand = cand[~cand.index.isin(used_rep)]
            if not cand.empty:
                m = cand.iloc[0]; used_rep.add(m.name)
                rows.append({'kind': 'matched', 'symbol': l['symbol'], 'side': l['side'],
                             'entry_time_utc': l['entry_time_utc'],
                             'live_pnl_usdt': l['pnl_usdt'], 'replay_pnl_usdt': m['pnl_usdt']})
            else:
                rows.append({'kind': 'live_only', 'symbol': l['symbol'], 'side': l['side'],
                             'entry_time_utc': l['entry_time_utc'],
                             'live_pnl_usdt': l['pnl_usdt'], 'replay_pnl_usdt': np.nan})
    if not replay.empty:
        for idx, r in replay.iterrows():
            if idx in used_rep:
                continue
            rows.append({'kind': 'replay_only', 'symbol': r['symbol'], 'side': r['side'],
                         'entry_time_utc': r['entry_time_utc'],
                         'live_pnl_usdt': np.nan, 'replay_pnl_usdt': r['pnl_usdt']})
    return pd.DataFrame(rows)

trade_match_df = match_live_vs_replay(live_trades_df, replay_trades_df)

def per_symbol_compare(live: pd.DataFrame, replay: pd.DataFrame) -> pd.DataFrame:
    syms = set()
    if not live.empty:   syms |= set(live['symbol'].unique())
    if not replay.empty: syms |= set(replay['symbol'].unique())
    matched_per = (trade_match_df[trade_match_df['kind'] == 'matched']
                   .groupby('symbol').size().to_dict()) if not trade_match_df.empty else {}
    rows = []
    for sym in sorted(syms):
        l = live  [live  ['symbol'] == sym] if not live.empty   else pd.DataFrame()
        r = replay[replay['symbol'] == sym] if not replay.empty else pd.DataFrame()
        ln, rn = len(l), len(r)
        l_wr = 100.0 * (l['pnl_usdt'] > 0).sum() / ln if ln else 0.0
        r_wr = 100.0 * (r['pnl_usdt'] > 0).sum() / rn if rn else 0.0
        rows.append({
            'symbol':          sym,
            'live_trades':     ln, 'replay_trades': rn,
            'matched':         int(matched_per.get(sym, 0)),
            'live_pnl_usdt':   round(float(l['pnl_usdt'].sum()), 2) if ln else 0.0,
            'replay_pnl_usdt': round(float(r['pnl_usdt'].sum()), 2) if rn else 0.0,
            'live_WR_%':       round(l_wr, 2),
            'replay_WR_%':     round(r_wr, 2),
        })
    return pd.DataFrame(rows)

compare_df = per_symbol_compare(live_trades_df, replay_trades_df)

print(f'Authoritative live trades: {len(live_trades_df)}   Replay trades: {len(replay_trades_df)}')
if not trade_match_df.empty:
    vc = trade_match_df['kind'].value_counts()
    n_match = int(vc.get('matched', 0))
    print('\nMatch breakdown (real trades vs backtest):')
    print(f"  matched     : {n_match:3d}  — real trades that ALSO appear in the backtest")
    print(f"  live_only   : {int(vc.get('live_only', 0)):3d}  — real trades with NO backtest twin")
    print(f"  replay_only : {int(vc.get('replay_only', 0)):3d}  — backtest trades never taken live")
    if len(live_trades_df):
        print(f"  → {100*n_match/len(live_trades_df):.0f}% of real trades have a backtest twin")
print('\nPer-symbol comparison (live = Bybit ledger truth):')
if compare_df.empty:
    print('  (no trades in either set)')
else:
    print(compare_df.to_string(index=False))

# Persist comparison artefacts alongside the ledger
if not trade_match_df.empty:
    trade_match_df.to_csv(ARTIFACTS_DIR / 'trade_match.csv', index=False)
if not compare_df.empty:
    compare_df.to_csv(ARTIFACTS_DIR / 'live_vs_replay_summary.csv', index=False)

Authoritative live trades: 115   Replay trades: 1381

Match breakdown (real trades vs backtest):
  matched     :  10  — real trades that ALSO appear in the backtest
  live_only   : 105  — real trades with NO backtest twin
  replay_only : 1371  — backtest trades never taken live
  → 9% of real trades have a backtest twin

Per-symbol comparison (live = Bybit ledger truth):
      symbol  live_trades  replay_trades  matched  live_pnl_usdt  replay_pnl_usdt  live_WR_%  replay_WR_%
     ADAUSDT           20            146        0     115.750000        19.300000  40.000000    30.820000
    AVAXUSDT           25            156        0      68.750000       424.380000  36.000000    34.620000
     BCHUSDT            3             36        0     131.590000       -39.700000 100.000000    27.780000
     BNBUSDT            0            103        0       0.000000       759.300000   0.000000    47.570000
    DOGEUSDT           17            168        0    -101.890000       729.800000  23.530000    

## Section 14 — Full replay trade ledger (sorted by entry_time_utc)

Every replay trade, time-ordered, no row truncation — for case-by-case inspection. If the table is empty there were no replay trades inside the window from Section 1.

In [ ]:
print(f'Full replay ledger — {len(replay_trades_df)} trades, sorted by entry_time_utc\n')
if replay_trades_df.empty:
    print('(no replay trades in this window)')
else:
    ledger = replay_trades_df.sort_values('entry_time_utc').reset_index(drop=True)
    with pd.option_context('display.max_rows', None,
                            'display.max_colwidth', 60):
        print(ledger.to_string(index=False))


Full replay ledger — 1381 trades, sorted by entry_time_utc

      symbol      entry_time_utc       exit_time_utc  side  entry_price  exit_price   stop_loss  take_profit      quantity  risk_usdt   pnl_usdt  pnl_usdt_gross     pnl_r exit_reason      signal_bar_utc
     XRPUSDT 2025-04-17 16:00:00 2025-04-21 00:00:00 short     2.066200    2.116912    2.116912     1.990132    394.384679  20.000000 -20.900000      -20.000000 -1.045000          SL 2025-04-17 16:00:00
     BCHUSDT 2025-04-17 20:00:00 2025-04-24 03:00:00  long   338.100000  368.498889  325.940445   368.498889      1.644797  20.000000  49.380000       50.000000  2.469000          TP 2025-04-17 20:00:00
     BNBUSDT 2025-04-18 03:00:00 2025-04-21 06:00:00  long   589.400000  608.123699  580.038151   608.123699      2.136330  20.000000  38.620000       40.000000  1.931000          TP 2025-04-18 03:00:00
     XLMUSDT 2025-04-19 09:00:00 2025-04-20 10:00:00  long     0.247170    0.239128    0.239128     0.267275   2486.944126  20.0